# MLFlow Tracking Server on Chameleon

This notebook sets up an MLFlow tracking server on Chameleon with:

- A VM instance at KVM@TACC
- Persistent block storage for PostgreSQL data
- Artifact storage using CHI@TACC S3 object storage

After running this notebook, you will have an MLFlow tracking server accessible via browser.

## Configuration

Set your project ID prefix. All resources (lease, server, volume, bucket) will be named with this prefix. Also set the lease time, i.e. how long do you want your MLFlow service to stay up? (Even after it ends, the data will persist to the next run. You should set the lease duration to the anticipated duration of the training experiments you plan to run, plus some headroom.)

In [1]:
# Set your project ID prefix (e.g., proj01, proj02, etc.)
# All resources will be named with this prefix
PROJECT_ID = "proj19"  # <-- Change this to your assigned project ID
LEASE_HOURS = 96 # Lease duration in hours (adjust as needed)

In [2]:
from chi import server, context, lease, network
import chi
import os
import datetime

## Create S3 Bucket and Credentials at CHI@TACC

MLFlow will store artifacts in CHI@TACC S3 object storage. We first create the bucket and credentials.

In [3]:
context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@TACC")

In [4]:
BUCKET_NAME = f"{PROJECT_ID}-mlflow-artifacts"

Create the S3 bucket if it doesn't already exist:

In [5]:
import swiftclient

os_conn = chi.clients.connection()
token = os_conn.authorize()
storage_url = os_conn.object_store.get_endpoint()

swift_conn = swiftclient.Connection(preauthurl=storage_url,
                                    preauthtoken=token,
                                    retries=5)

# Check if bucket exists, create if not
try:
    swift_conn.head_container(BUCKET_NAME)
    print(f"Bucket {BUCKET_NAME} already exists")
except swiftclient.exceptions.ClientException:
    swift_conn.put_container(BUCKET_NAME)
    print(f"Created bucket {BUCKET_NAME}")

Created bucket proj19-mlflow-artifacts


Generate EC2 credentials for S3 access. These will be saved to a file so they can be reused.


In [6]:
ENV_FILE = os.path.expanduser("~/work/.mlflow_s3_credentials")

if os.path.exists(ENV_FILE):
    # Load existing credentials
    credentials = {}
    with open(ENV_FILE, "r") as f:
        for line in f:
            line = line.strip()
            if "=" in line and not line.startswith("#"):
                key, value = line.split("=", 1)
                credentials[key] = value
    AWS_ACCESS_KEY_ID = credentials["AWS_ACCESS_KEY_ID"]
    AWS_SECRET_ACCESS_KEY = credentials["AWS_SECRET_ACCESS_KEY"]
    print(f"Loaded existing credentials from {ENV_FILE}")
else:
    # Generate new credentials
    conn = chi.clients.connection()
    project_id = conn.current_project_id
    identity_ep = conn.session.get_endpoint(service_type="identity", interface="public")
    url = f"{identity_ep}/v3/users/{conn.current_user_id}/credentials/OS-EC2"
    
    resp = conn.session.post(url, json={"tenant_id": project_id})
    resp.raise_for_status()
    ec2 = resp.json()["credential"]
    
    AWS_ACCESS_KEY_ID = ec2["access"]
    AWS_SECRET_ACCESS_KEY = ec2["secret"]
    
    # Save credentials to file
    os.makedirs(os.path.dirname(ENV_FILE), exist_ok=True)
    with open(ENV_FILE, "w") as f:
        f.write(f"AWS_ACCESS_KEY_ID={AWS_ACCESS_KEY_ID}\n")
        f.write(f"AWS_SECRET_ACCESS_KEY={AWS_SECRET_ACCESS_KEY}\n")
    print(f"Generated new credentials and saved to {ENV_FILE}")

print(f"Bucket: {BUCKET_NAME}")

Generated new credentials and saved to /home/am16455_nyu_edu/work/.mlflow_s3_credentials
Bucket: proj19-mlflow-artifacts


## Create Lease and Server at KVM@TACC

In [7]:
context.choose_site(default="KVM@TACC")

Create a 12-hour lease for an m1.medium VM:

In [8]:
l = lease.Lease(f"{PROJECT_ID}-mlflow-lease", duration=datetime.timedelta(hours=LEASE_HOURS))
l.add_flavor_reservation(id=chi.server.get_flavor_id("m1.medium"), amount=1)
l.submit(idempotent=True)

Waiting for lease to start...


Lease proj19-mlflow-lease has reached status active


In [9]:
l.show()

HTML(value='\n        <h2>Lease Details</h2>\n        <table>\n            <tr><th>Name</th><td>proj19-mlflow-…

Lease Details:
Name: proj19-mlflow-lease
ID: b809c28b-ff5c-4110-a25a-358d1195aba0
Status: ACTIVE
Start Date: 2026-04-03 17:55:00
End Date: 2026-04-07 17:55:00
User ID: ade47bc333842e91badf68ffe0bfafaef29bde4e87c40ea065f221dfcb80a8ad
Project ID: 89f528973fea4b3a981f9b2344e522de

Node Reservations:

Floating IP Reservations:

Network Reservations:

Flavor Reservations:
ID: d2094f7b-bd1d-4e0c-97e4-dd832c94df3c, Status: active, Flavor: d2094f7b-bd1d-4e0c-97e4-dd832c94df3c, Amount: 1

Events:


Launch the VM instance:

In [10]:
s = server.Server(
    f"{PROJECT_ID}-mlflow-server",
    image_name="CC-Ubuntu24.04",
    flavor_name=l.get_reserved_flavors()[0].name
)
s.submit(idempotent=True)

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Waiting for server proj19-mlflow-server's status to become ACTIVE. This typically takes 10 minutes for baremetal, but can take up to 20 minutes.


Server has moved to status ACTIVE


Attribute,proj19-mlflow-server
Id,9006c39e-4fe1-453e-840f-eec050a7f43c
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:d2094f7b-bd1d-4e0c-97e4-dd832c94df3c
Addresses,sharednet1: IP: 10.56.3.244 (v4) Type: fixed MAC: fa:16:3e:af:0c:14
Network Name,sharednet1
Created At,2026-04-03T17:56:52Z
Keypair,am16455_nyu_edu-jupyter
Reservation Id,None
Host Id,d6959d764fb6c3a2f8d4af1e101282f4e575e3f34464fe6cb24c605f


Associate a floating IP:

In [12]:
s.associate_floating_ip()

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


ResourceError: None of the ports can route to floating ip 129.114.26.23 on server 9006c39e-4fe1-453e-840f-eec050a7f43c

Set up security groups to allow SSH, MLFlow UI (port 8000):

In [13]:
security_groups = [
    {"name": "allow-ssh", "port": 22, "description": "Enable SSH traffic on TCP port 22"},
    {"name": "allow-8000", "port": 8000, "description": "Enable TCP port 8000 (MLFlow UI)"}
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        "name": sg["name"],
        "description": sg["description"],
    })
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"Updated security groups: {[sg['name'] for sg in security_groups]}")

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Updated security groups: ['allow-ssh', 'allow-8000']


In [14]:
s.refresh()
s.check_connectivity()

Checking connectivity to 129.114.26.107 port 22.


Connection successful


In [15]:
s.refresh()
s.show(type="widget")

Attribute,proj19-mlflow-server
Id,9006c39e-4fe1-453e-840f-eec050a7f43c
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:d2094f7b-bd1d-4e0c-97e4-dd832c94df3c
Addresses,sharednet1: IP: 10.56.3.244 (v4) Type: fixed MAC: fa:16:3e:af:0c:14 IP: 129.114.26.107 (v4) Type: floating MAC: fa:16:3e:af:0c:14
Network Name,sharednet1
Created At,2026-04-03T17:56:52Z
Keypair,am16455_nyu_edu-jupyter
Reservation Id,None
Host Id,d6959d764fb6c3a2f8d4af1e101282f4e575e3f34464fe6cb24c605f


## Set up Persistent Volume

We will use a persistent block storage volume for PostgreSQL data. If the volume does not exist, we create it.

In [16]:
VOLUME_NAME = f"{PROJECT_ID}-mlflow-persist"
VOLUME_SIZE_GB = 5

In [17]:
cinder_client = chi.clients.cinder()

# Check if volume already exists
existing_volumes = [v for v in cinder_client.volumes.list() if v.name == VOLUME_NAME]

if existing_volumes:
    volume = existing_volumes[0]
    print(f"Found existing volume: {volume.name} (status: {volume.status})")
else:
    volume = cinder_client.volumes.create(name=VOLUME_NAME, size=VOLUME_SIZE_GB)
    print(f"Created new volume: {volume.name}")

Created new volume: proj19-mlflow-persist


In [18]:
# Wait for volume to be available if just created
import time
volume = cinder_client.volumes.get(volume.id)
while volume.status not in ["available", "in-use"]:
    print(f"Volume status: {volume.status}, waiting...")
    time.sleep(5)
    volume = cinder_client.volumes.get(volume.id)
print(f"Volume status: {volume.status}")

Volume status: available


Attach the volume to our server (if not already attached):

In [19]:
volume = cinder_client.volumes.get(volume.id)
if volume.status == "available":
    s.attach_volume(volume.id)
    print(f"Attached volume {VOLUME_NAME} to server")
else:
    print(f"Volume status is {volume.status}, skipping attach")

Attached volume proj19-mlflow-persist to server


## Set up Docker on the Server

In [20]:
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")

/opt/conda/lib/python3.12/site-packages/paramiko/client.py:885: UserWarning: Unknown ssh-ed25519 host key for 129.114.26.107: b'c69a598ab118b633a1cbfe8b8f4b3936'
  warnings.warn(


# Executing docker install script, commit: c04fb16bb8bd8ed6ce884bb40570cbcd6101ae0c


+ sh -c apt-get -qq update >/dev/null
+ sh -c DEBIAN_FRONTEND=noninteractive apt-get -y -qq install ca-certificates curl >/dev/null

Running kernel seems to be up-to-date.

Restarting services...
 systemctl restart packagekit.service

No containers need to be restarted.

No user sessions are running outdated binaries.

No VM guests are running outdated hypervisor (qemu) binaries on this host.
+ sh -c install -m 0755 -d /etc/apt/keyrings
+ sh -c curl -fsSL "https://download.docker.com/linux/ubuntu/gpg" -o /etc/apt/keyrings/docker.asc
+ sh -c chmod a+r /etc/apt/keyrings/docker.asc
+ sh -c echo "deb [arch=amd64 signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu noble stable" > /etc/apt/sources.list.d/docker.list
+ sh -c apt-get -qq update >/dev/null
+ sh -c DEBIAN_FRONTEND=noninteractive apt-get -y -qq install docker-ce docker-ce-cli containerd.io docker-compose-plugin docker-ce-rootless-extras docker-buildx-plugin docker-model-plugin >/dev/null

Running kern

  UNIT                                                                           LOAD   ACTIVE SUB       DESCRIPTION
  proc-sys-fs-binfmt_misc.automount                                              loaded active running   Arbitrary Executable File Formats File System Automount Point
  sys-devices-pci0000:00-0000:00:03.0-virtio1-net-ens3.device                    loaded active plugged   Virtio network device
  sys-devices-pci0000:00-0000:00:04.0-virtio2-block-vda-vda1.device              loaded active plugged   /sys/devices/pci0000:00/0000:00:04.0/virtio2/block/vda/vda1
  sys-devices-pci0000:00-0000:00:04.0-virtio2-block-vda-vda2.device              loaded active plugged   /sys/devices/pci0000:00/0000:00:04.0/virtio2/block/vda/vda2
  sys-devices-pci0000:00-0000:00:04.0-virtio2-block-vda-vda3.device              loaded active plugged   /sys/devices/pci0000:00/0000:00:04.0/virtio2/block/vda/vda3
  sys-devices-pci0000:00-0000:00:04.0-virtio2-block-vda.device                   loaded active

INFO: Docker daemon enabled and started



0/serial8250:0/serial8250:0.29/tty/ttyS29
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.3-tty-ttyS3.device   loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.3/tty/ttyS3
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.30-tty-ttyS30.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.30/tty/ttyS30
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.31-tty-ttyS31.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.31/tty/ttyS31
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.4-tty-ttyS4.device   loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.4/tty/ttyS4
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.5-tty-ttyS5.device   loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.5/tty/ttyS5
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.6-tty-ttyS6.de

+ sh -c docker version


vice   loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.6/tty/ttyS6
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.7-tty-ttyS7.device   loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.7/tty/ttyS7
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.8-tty-ttyS8.device   loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.8/tty/ttyS8
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.9-tty-ttyS9.device   loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.9/tty/ttyS9
  sys-devices-pnp0-00:00-00:00:0-00:00:0.0-tty-ttyS0.device                      loaded active plugged   /sys/devices/pnp0/00:00/00:00:0/00:00:0.0/tty/ttyS0
  sys-devices-virtual-misc-rfkill.device                                         loaded active plugged   /sys/devices/virtual/misc/rfkill
  sys-devices-virtual-net-docker0.device                                   

<Result cmd='sudo groupadd -f docker; sudo usermod -aG docker $USER' exited=0>

## Mount the Persistent Volume

Format and mount the volume. If the volume already has data, we skip formatting.

In [21]:
# Check if volume needs formatting (new volume)
# Get the real device path for this attached volume
volume = cinder_client.volumes.get(volume.id)
device = volume.attachments[0]["device"]   # e.g. /dev/vdb
part = f"{device}1"
s.execute("lsblk")

NAME   MAJ:MIN RM  SIZE RO TYPE MOUNTPOINTS
vda    253:0    0   40G  0 disk 
├─vda1 253:1    0  550M  0 part /boot/efi
├─vda2 253:2    0    8M  0 part 
└─vda3 253:3    0 39.5G  0 part /
vdb    253:16   0    5G  0 disk 


<Result cmd='lsblk' exited=0>

In [22]:
# Check if partition exists - if not, create it and format
# This will only format if the volume is new (no partition table)
s.execute(f"""
set -e
if ! sudo blkid {part} >/dev/null 2>&1; then
    echo "No filesystem found on {part}; creating partition + ext4"
    sudo parted -s {device} mklabel gpt
    sudo parted -s {device} mkpart primary ext4 0% 100%
    sudo mkfs.ext4 {part}
else
    echo "Filesystem already exists on {part}; skipping format"
fi
sudo mkdir -p /mnt/mlflow_persist
if ! mountpoint -q /mnt/mlflow_persist; then
    sudo mount {part} /mnt/mlflow_persist
fi
sudo chown -R cc:cc /mnt/mlflow_persist
""")

No filesystem found on /dev/vdb1; creating partition + ext4


mke2fs 1.47.0 (5-Feb-2023)


Discarding device blocks:       0/131020              done                            
Creating filesystem with 1310208 4k blocks and 327680 inodes
Filesystem UUID: 2109f3c5-76cb-44e9-93c0-c6ce690a51d5
Superblock backups stored on blocks: 
	32768, 98304, 163840, 229376, 294912, 819200, 884736

Allocating group tables:  0/4    done                            
Writing inode tables:  0/4    done                            
Creating journal (16384 blocks): done
Writing superblocks and filesystem accounting information:  0/4    done



<Result cmd='\nset -e\nif ! sudo blkid /dev/vdb1 >/dev/null 2>&1; then\n    echo "No filesystem found on /dev/vdb1; creating partition + ext4"\n    sudo parted -s /dev/vdb mklabel gpt\n    sudo parted -s /dev/vdb mkpart primary ext4 0% 100%\n    sudo mkfs.ext4 /dev/vdb1\nelse\n    echo "Filesystem already exists on /dev/vdb1; skipping format"\nfi\nsudo mkdir -p /mnt/mlflow_persist\nif ! mountpoint -q /mnt/mlflow_persist; then\n    sudo mount /dev/vdb1 /mnt/mlflow_persist\nfi\nsudo chown -R cc:cc /mnt/mlflow_persist\n' exited=0>

In [23]:
# Mount the volume
s.execute("""
sudo mkdir -p /mnt/mlflow_persist
sudo mount /dev/vdb1 /mnt/mlflow_persist
sudo chown -R cc:cc /mnt/mlflow_persist
""")

mount: /mnt/mlflow_persist: /dev/vdb1 already mounted on /mnt/mlflow_persist.
       dmesg(1) may have more information after failed mount system call.


<Result cmd='\nsudo mkdir -p /mnt/mlflow_persist\nsudo mount /dev/vdb1 /mnt/mlflow_persist\nsudo chown -R cc:cc /mnt/mlflow_persist\n' exited=0>

In [24]:
# Create directory for postgres data
s.execute("mkdir -p /mnt/mlflow_persist/postgres_data")

<Result cmd='mkdir -p /mnt/mlflow_persist/postgres_data' exited=0>

## Transfer Docker Files to Server

In [25]:
# Create docker directory on server
s.execute("mkdir -p ~/docker")

<Result cmd='mkdir -p ~/docker' exited=0>

In [26]:
# Create .env file with S3 credentials
env_content = f"""AWS_ACCESS_KEY_ID={AWS_ACCESS_KEY_ID}
AWS_SECRET_ACCESS_KEY={AWS_SECRET_ACCESS_KEY}
BUCKET_NAME={BUCKET_NAME}
"""

with open("docker/.env", "w") as f:
    f.write(env_content)
print("Created docker/.env")

Created docker/.env


In [34]:
# Upload docker-compose and .env to server
s.upload("docker/docker-compose-mlflow.yaml", "/home/cc/docker/docker-compose-mlflow.yaml")
s.upload("docker/.env", "/home/cc/docker/.env")
print("Uploaded docker files to server")

Uploaded docker files to server


In [35]:
# Verify the files were uploaded
s.execute("ls -la ~/docker/")

total 16
drwxrwxr-x 2 cc cc 4096 Apr  3 18:06 .
drwxr-x--- 5 cc cc 4096 Apr  3 18:02 ..
-rw-r--r-- 1 cc cc  142 Apr  3 18:10 .env
-rw-r--r-- 1 cc cc 1062 Apr  3 18:10 docker-compose-mlflow.yaml


<Result cmd='ls -la ~/docker/' exited=0>

## Start MLFlow

Start the MLFlow tracking server with Docker Compose:

In [36]:
s.execute("docker compose -f ~/docker/docker-compose-mlflow.yaml up -d")

 Image postgres:18 Pulling 
 Image ghcr.io/mlflow/mlflow:v3.9.0 Pulling 
 f3bba6db66bc Pulling fs layer 0B
 ec781dee3f47 Pulling fs layer 0B
 ce2467f2f21d Pulling fs layer 0B
 d4ced18622af Pulling fs layer 0B
 af73ff344a10 Pulling fs layer 0B
 07cec992154a Pulling fs layer 0B
 fd0dcffac314 Pulling fs layer 0B
 bda354b903ce Pulling fs layer 0B
 fcd98f4944fb Pulling fs layer 0B
 917d5439b698 Pulling fs layer 0B
 daebffc75218 Pulling fs layer 0B
 895ed9837058 Pulling fs layer 0B
 365add159c69 Pulling fs layer 0B
 7fcf5ac96429 Pulling fs layer 0B
 ccaf924377f9 Pulling fs layer 0B
 5e584f8f28c3 Pulling fs layer 0B
 954774345a61 Pulling fs layer 0B
 7360b233c860 Pulling fs layer 0B
 af9063ecbdef Download complete 0B
 07cec992154a Downloading 1.049MB
 ce2467f2f21d Download complete 0B
 b26b36c837dc Downloading 3.146MB
 f3bba6db66bc Download complete 0B
 bda354b903ce Download complete 0B
 07cec992154a Downloading 3.146MB
 917d5439b698 Downloading 1.049MB
 7360b233c860 Download complete 0B
 2e0

<Result cmd='docker compose -f ~/docker/docker-compose-mlflow.yaml up -d' exited=0>

In [37]:
# Check that containers are running
s.execute("docker ps")

CONTAINER ID   IMAGE                          COMMAND                  CREATED         STATUS         PORTS                                         NAMES
324df122f0f5   ghcr.io/mlflow/mlflow:v3.9.0   "/bin/sh -c 'pip ins…"   3 seconds ago   Up 2 seconds   0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp   mlflow
7ce249351ebc   postgres:18                    "docker-entrypoint.s…"   4 seconds ago   Up 2 seconds   0.0.0.0:5432->5432/tcp, [::]:5432->5432/tcp   postgres


<Result cmd='docker ps' exited=0>

## Access MLFlow UI

Get the floating IP and access MLFlow:

In [38]:
s.refresh()
floating_ip = s.get_floating_ip()
print(f"MLFlow UI: http://{floating_ip}:8000")
print(f"\nSSH access: ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip}")
print(f"\nTo log experiments to this MLFlow server from another machine, set:")
print(f"  export MLFLOW_TRACKING_URI=http://{floating_ip}:8000")

MLFlow UI: http://129.114.26.107:8000

SSH access: ssh -i ~/.ssh/id_rsa_chameleon cc@129.114.26.107

To log experiments to this MLFlow server from another machine, set:
  export MLFLOW_TRACKING_URI=http://129.114.26.107:8000


## Cleanup (Optional)

When you are done, you can stop the containers and optionally delete resources.

In [ ]:
# Stop containers
# s.execute("docker compose -f ~/docker/docker-compose-mlflow.yaml down")

In [ ]:
# Unmount volume before detaching
# s.execute("sudo umount /mnt/mlflow_persist")

In [ ]:
# Detach volume (preserves data for next time)
# volume = cinder_client.volumes.get(volume.id)
# s.detach_volume(volume.id)

In [ ]:
# Delete server (preserves volume)
# s.delete()

In [ ]:
# Delete lease
# l.delete()